**Pre Trained ResNet34**

**Libraries**

In [1]:
import sys
from pathlib import Path

cloned_rep_path=r'C:\Users\laika\OneDrive\Documents\Capstone I All Files\Capstone-DATA4381'
sys.path.insert(0,cloned_rep_path)

from capstone_data.image_loader import load_dataset, split_dataset, load_arrays

from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import resnet34, ResNet34_Weights


**Load Dataset**

In [ ]:
# Dictionary to store the datasets

train_v1 = load_dataset('aug_data_v1', split='train')
val_v1 = load_dataset('aug_data_v1', split='validation')
test_v1 = load_dataset('aug_data_v1', split='test')

train_v3 = load_dataset('aug_data_v3', split='train')
val_v3 = load_dataset('aug_data_v3', split='validation')
test_v3 = load_dataset('aug_data_v3', split='test')

print(len(train_v1))
print(len(val_v1))
print(len(test_v1))

print(len(train_v3))
print(len(val_v3))
print(len(test_v3))

aug_dict={
    'aug_data_v1': {'train': train_v1, 'val': val_v1, 'test': test_v1},
    'aug_data_v3': {'train': train_v3, 'val': val_v3, 'test': test_v3}
}

14019
1662
1673
12047
1650
1646
dict_items([('aug_data_v1', {'train': [{'path': WindowsPath('C:/Users/laika/OneDrive/Documents/Capstone I All Files/Capstone-DATA4381/data/augmented_data/aug_data_v1/angry/train/augmented_000000.jpg'), 'label': 0, 'class_name': 'angry', 'source': 'aug_data_v1'}, {'path': WindowsPath('C:/Users/laika/OneDrive/Documents/Capstone I All Files/Capstone-DATA4381/data/augmented_data/aug_data_v1/angry/train/augmented_000001.jpg'), 'label': 0, 'class_name': 'angry', 'source': 'aug_data_v1'}, {'path': WindowsPath('C:/Users/laika/OneDrive/Documents/Capstone I All Files/Capstone-DATA4381/data/augmented_data/aug_data_v1/angry/train/augmented_000002.jpg'), 'label': 0, 'class_name': 'angry', 'source': 'aug_data_v1'}, {'path': WindowsPath('C:/Users/laika/OneDrive/Documents/Capstone I All Files/Capstone-DATA4381/data/augmented_data/aug_data_v1/angry/train/augmented_000003.jpg'), 'label': 0, 'class_name': 'angry', 'source': 'aug_data_v1'}, {'path': WindowsPath('C:/Users/la

In [21]:
class DataManger:
    def __init__(self, aug_dict, image_size=48, batch_size=32):
        self.aug_dict = aug_dict
        self.batch_size=batch_size
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
        
    def image_preprocessing(self):  
        train_proc = {} 
             
        for key, values in self.aug_dict.items():
            train_proc[key] = {} 
               
            for split, records in values.items():
                samples = []

                for record in records:
                    with Image.open(record["path"]) as image:
                        image_tensor = self.transform(image)

                    samples.append((image_tensor, record["label"]))

                train_proc[key][split] = DataLoader(
                    samples,
                    batch_size=self.batch_size,
                    shuffle=(split == "train"),
                )

        return train_proc
    
manage=DataManger(aug_dict)
load=manage.image_preprocessing()

train_new_v1=load['aug_data_v1']['train']
train_new_v3=load['aug_data_v3']['train']

test_new_v1=load['aug_data_v1']['test']
test_new_v3=load['aug_data_v3']['test']

val_new_v1=load['aug_data_v1']['val']
val_new_v3=load['aug_data_v3']['val']

print(f'Length of new train v1: {len(train_new_v1)}')
print(f'Length of new train v3: {len(train_new_v3)}')
print(f'Length of new test v1: {len(test_new_v1)}')
print(f'Length of new test v3: {len(test_new_v3)}')
print(f'Length of new val v1: {len(val_new_v1)}')
print(f'Length of new val v3: {len(val_new_v3)}')

Length of new train v1: 439
Length of new train v3: 377
Length of new test v1: 53
Length of new test v3: 52
Length of new val v1: 52
Length of new val v3: 52


In [25]:
# Datasets are now torch objects .dataset()
data_names=[train_new_v1, train_new_v3, test_new_v1, test_new_v3, val_new_v1, val_new_v3]
for i in data_names:
    print(f'Length of dataset {i}: {len(i.dataset)}')


Length of dataset <torch.utils.data.dataloader.DataLoader object at 0x0000021562D85290>: 14019
Length of dataset <torch.utils.data.dataloader.DataLoader object at 0x000002155BE49A10>: 12047
Length of dataset <torch.utils.data.dataloader.DataLoader object at 0x0000021559A83C10>: 1673
Length of dataset <torch.utils.data.dataloader.DataLoader object at 0x000002155F257910>: 1646
Length of dataset <torch.utils.data.dataloader.DataLoader object at 0x0000021560C6D150>: 1662
Length of dataset <torch.utils.data.dataloader.DataLoader object at 0x000002155BE5F190>: 1650


In [26]:
# New dictionary to store the processed datasets
new_dict = {
    'aug_v1': {'train': train_new_v1, 'val': val_new_v1, 'test': test_new_v1},
    'aug_v3': {'train': train_new_v3, 'val': val_new_v3, 'test': test_new_v3}
}

**Modify Resnet34 model**

In [30]:
# Modify ResNet34 for 3 classes & 48x48  images

def modify_resnet34(num_classes=3):
    
    model=resnet34(weights=None) # No weights, since default weights expect 224x224 colored images
    
    # Modify the first layer for 48,48 images
    model.conv1=nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    
    # Remove maxpool layer, to avoid shrinking image features too much
    model.maxpool=nn.Identity()
    
    # Final layer
    model.fc=nn.Linear(model.fc.in_features,num_classes)
    
    return model

In [ ]:
# Initialize the model

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=modify_resnet34(num_classes=3).to(device)

In [20]:
# Test Batch

test_batch=aug_dict['aug_data_v1']['train']
images,labels=next(iter(test_batch))
output=model(images)

print(f'Input batch shape: {images.shape}')
print(f'Output shape: {output.shape}')

Input batch shape: torch.Size([32, 3, 48, 48])
Output shape: torch.Size([32, 3])


**Train the Model**

In [31]:

class ResNet34Trainer:
    def __init__(self, aug_dict, num_classes=3, lr=0.001, device=None):
        self.aug_dict=aug_dict
        self.num_classes=num_classes
        self.lr=lr
        if device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = device
        self.trained_versions={}
    
    def train_model(self,num_epochs=3):
        
        # Iterate through all versions
        for name, data in self.aug_dict.items():
            print(f'Training: {name}')
            
            model=modify_resnet34(num_classes=self.num_classes).to(self.device)
            
            # Loss function
            criterion=nn.CrossEntropyLoss()
            # Optimizer
            optimizer=optim.Adam(model.parameters(), lr=self.lr)
            
            train_data=data['train']
            
            # Epochs
            for epoch in range(num_epochs):
                model.train()
                total_loss=0.0 #total loss per batch
                
                for images,labels in train_data:
                    
                    # Ensure images and labels are both on either CPU or GPU
                    images,labels=images.to(self.device),labels.to(self.device)
                    
                    # Add up gradients from previous step
                    optimizer.zero_grad()
                    
                    # Forward pass
                    outputs=model(images)
                    
                    # Compute loss (Compare model's predions with the true labels)
                    loss=criterion(outputs, labels)

                    # Backpropagation
                    loss.backward()
                    
                    # Update weights
                    optimizer.step()
                    
                    total_loss+=loss.item()
                
                avg_loss=total_loss/len(train_data)
                print(f'Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss}')
                # check test/val loss
            # Store model version
            self.trained_versions[name]=model
            print(f'Training completed for {name}')
            
        return self.trained_versions

In [32]:
train=ResNet34Trainer(new_dict, num_classes=3, lr=0.001)
trained_models=train.train_model(num_epochs=3)

Training: aug_v1


KeyboardInterrupt: 